In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split


In [ ]:
#Punto 2 de la rubrica
# =====================================================
# 1. CARGAR DATASET
# =====================================================

# Cargar el archivo CSV indicando el separador correcto
df = pd.read_csv("sonrievalencia_citas_transformado.csv", sep=";")

print("Dataset cargado correctamente")


# =====================================================
# 2. PRIMERAS FILAS
# =====================================================

# Mostrar las primeras 5 filas
display(df.head())


# =====================================================
# 3. DIMENSIONES DEL DATASET
# =====================================================

# Comprobar número de filas y columnas
print("Dimensiones del dataset:", df.shape)

print("Número de filas:", df.shape[0])
print("Número de columnas:", df.shape[1])


# =====================================================
# 4. INFORMACIÓN GENERAL
# =====================================================

# Mostrar tipos de datos y valores no nulos
df.info()


# =====================================================
# 5. ESTADÍSTICAS DESCRIPTIVAS
# =====================================================

# Estadísticas de todas las columnas
display(df.describe(include="all"))


# =====================================================
# 6. VALORES NULOS REALES
# =====================================================

# Contar los valores nulos de cada columna
print("Valores nulos por columna:")

display(df.isnull().sum())

# Porcentaje de valores nulos
print("Porcentaje de valores nulos:")

display((df.isnull().mean() * 100).round(2))


# =====================================================
# 7. DETECCIÓN DE NULOS DISFRAZADOS
# =====================================================

# Valores que podrían representar datos ausentes
valores_nulos = [
    "-", "?", "N/A", "NA", "NULL",
    "NONE", "DESCONOCIDO", "NO DISPONIBLE"
]

print("BÚSQUEDA DE POSIBLES NULOS DISFRAZADOS")

for columna in df.select_dtypes(include="object").columns:

    valores = df[columna].astype("string").str.strip().str.upper()

    encontrados = valores.isin(valores_nulos).sum()

    if encontrados > 0:
        print(f"{columna}: {encontrados} posibles nulos disfrazados")


# =====================================================
# 8. DETECCIÓN DE ESPACIOS EN BLANCO
# =====================================================

print("BÚSQUEDA DE ESPACIOS AL PRINCIPIO O FINAL")

for columna in df.select_dtypes(include="object").columns:

    espacios = df[columna].astype("string").str.contains(
        r"^\s|\s$", na=False
    ).sum()

    if espacios > 0:
        print(f"{columna}: {espacios} valores con espacios")


# =====================================================
# 9. VALORES ÚNICOS (unique)
# =====================================================

print("VALORES ÚNICOS DE CADA COLUMNA")

for columna in df.columns:

    print(f"\n--- {columna} ---")

    print(df[columna].unique())


# =====================================================
# 10. FRECUENCIA DE VALORES (value_counts)
# =====================================================

print("FRECUENCIA DE VALORES POR COLUMNA")

for columna in df.columns:

    print(f"\n--- {columna} ---")

    print(df[columna].value_counts(dropna=False))


# =====================================================
# 11. RESUMEN FINAL
# =====================================================

print("\n========== RESUMEN DEL DATASET ==========")

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])
print("Total de valores nulos:", df.isnull().sum().sum())
print("Columnas con valores nulos:", df.isnull().gt(0).sum())
print("Duplicados:", df.duplicated().sum())

print("=========================================")

In [ ]:

# =====================================================
# 1. COPIA DEL DATASET
# =====================================================

df_reducido = df.copy()

print("DIMENSIONES INICIALES:", df_reducido.shape)


# =====================================================
# 2. ELIMINAR COLUMNAS CONSTANTES
# =====================================================

columnas_constantes = [
    columna for columna in df_reducido.columns
    if df_reducido[columna].nunique(dropna=False) <= 1
]

print("\nCOLUMNAS CONSTANTES:")
print(columnas_constantes)

df_reducido.drop(
    columns=columnas_constantes,
    inplace=True
)


# =====================================================
# 3. ELIMINAR COLUMNAS DUPLICADAS
# =====================================================

columnas_duplicadas = df_reducido.T[
    df_reducido.T.duplicated()
].index.tolist()

print("\nCOLUMNAS DUPLICADAS:")
print(columnas_duplicadas)

df_reducido.drop(
    columns=columnas_duplicadas,
    inplace=True
)


# =====================================================
# 4. ELIMINAR COLUMNAS REDUNDANTES
# =====================================================

columnas_redundantes = []

# Comprobar si clinica y cod_clinica representan
# exactamente la misma información

if {"clinica", "cod_clinica"}.issubset(df_reducido.columns):

    equivalencia = (
        df_reducido.groupby("clinica")["cod_clinica"].nunique().max() == 1
        and
        df_reducido.groupby("cod_clinica")["clinica"].nunique().max() == 1
    )

    if equivalencia:
        columnas_redundantes.append("cod_clinica")


# Comprobar si el importe con IVA se calcula
# a partir del presupuesto sin IVA (21 %)

if {
    "importe_presupuesto_eur",
    "importe_con_iva_eur"
}.issubset(df_reducido.columns):

    base = df_reducido["importe_presupuesto_eur"]
    iva = df_reducido["importe_con_iva_eur"]

    validos = base.notna() & iva.notna()

    if validos.any():

        coincidencias = np.isclose(
            iva[validos],
            base[validos] * 1.21,
            atol=0.02
        )

        porcentaje = coincidencias.mean()

        print("\nCoincidencia entre importes:", round(porcentaje * 100, 2), "%")

        if porcentaje >= 0.99:
            columnas_redundantes.append("importe_con_iva_eur")


print("\nCOLUMNAS REDUNDANTES:")
print(columnas_redundantes)

df_reducido.drop(
    columns=columnas_redundantes,
    inplace=True
)


# =====================================================
# 5. ELIMINAR IDENTIFICADORES
# =====================================================

identificadores = [
    "id_cita",
    "dni"
]

identificadores_eliminados = [
    columna for columna in identificadores
    if columna in df_reducido.columns
]

print("\nIDENTIFICADORES EXCLUIDOS:")
print(identificadores_eliminados)

df_reducido.drop(
    columns=identificadores_eliminados,
    inplace=True
)


# =====================================================
# 6. DETECTAR COLUMNAS CORRELACIONADAS
# =====================================================

columnas_numericas = df_reducido.select_dtypes(
    include=np.number
).columns.tolist()

correlacion = df_reducido[columnas_numericas].corr().abs()

# Evitar duplicados en la matriz
mascara = np.triu(
    np.ones(correlacion.shape),
    k=1
).astype(bool)

correlaciones_altas = (
    correlacion.where(mascara)
    .stack()
)

print("\nCORRELACIONES SUPERIORES A 0.90:")

print(
    correlaciones_altas[
        correlaciones_altas > 0.90
    ]
)

columnas_correlacionadas = set()

for (columna1, columna2), valor in correlaciones_altas.items():

    if valor > 0.90:

        # Mantener la primera columna y eliminar la segunda
        columnas_correlacionadas.add(columna2)

print("\nCOLUMNAS ELIMINADAS POR CORRELACIÓN:")
print(sorted(columnas_correlacionadas))

df_reducido.drop(
    columns=list(columnas_correlacionadas),
    inplace=True
)


# =====================================================
# 7. RESUMEN DE LA REDUCCIÓN
# =====================================================

print("\n========== RESUMEN DE REDUCCIÓN ==========")

print("Dimensiones originales:", df.shape)
print("Dimensiones reducidas:", df_reducido.shape)

print("Columnas constantes:", columnas_constantes)
print("Columnas duplicadas:", columnas_duplicadas)
print("Columnas redundantes:", columnas_redundantes)
print("Identificadores excluidos:", identificadores_eliminados)
print("Columnas correlacionadas:", sorted(columnas_correlacionadas))

print("\nColumnas finales:")
print(df_reducido.columns.tolist())


# =====================================================
# 8. DETECTAR DESBALANCEO
# =====================================================

print("\n========== DESBALANCEO DE CLASES ==========")

objetivo = "no_acude"

if objetivo not in df_reducido.columns:
    raise ValueError("No se encuentra la columna no_acude")

print("\nFrecuencia de cada clase:")

frecuencias = df_reducido[objetivo].value_counts()

print(frecuencias)

print("\nPorcentajes:")

print(
    (df_reducido[objetivo].value_counts(normalize=True) * 100).round(2)
)

diferencia = frecuencias.max() / frecuencias.min()

print("\nProporción entre clases:", round(diferencia, 2))

if diferencia > 1.5:
    print("DESBALANCEO DETECTADO")
else:
    print("No se detecta un desbalanceo importante")


# =====================================================
# 9. SEPARAR VARIABLES Y OBJETIVO
# =====================================================

X = df_reducido.drop(columns=[objetivo])

y = df_reducido[objetivo]

# Dividir en entrenamiento y prueba
# Mantener la proporción original con stratify

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# =====================================================
# 10. EQUILIBRAR CLASES (OVERSAMPLING)
# =====================================================

print("\n========== EQUILIBRADO DE CLASES ==========")

print("Antes del equilibrado:")

print(y_train.value_counts())


# Unir temporalmente variables y objetivo
datos_train = X_train.copy()

datos_train[objetivo] = y_train

# Identificar clase mayoritaria y minoritaria
mayoritaria = y_train.value_counts().idxmax()
minoritaria = y_train.value_counts().idxmin()

# Separar ambas clases
datos_mayoritaria = datos_train[
    datos_train[objetivo] == mayoritaria
]

datos_minoritaria = datos_train[
    datos_train[objetivo] == minoritaria
]

# Aumentar la clase minoritaria mediante repetición
datos_minoritaria_equilibrados = datos_minoritaria.sample(
    n=len(datos_mayoritaria),
    replace=True,
    random_state=42
)

# Unir las dos clases
datos_equilibrados = pd.concat([
    datos_mayoritaria,
    datos_minoritaria_equilibrados
])

# Mezclar los registros
datos_equilibrados = datos_equilibrados.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


# Recuperar X e y equilibrados
X_train_equilibrado = datos_equilibrados.drop(
    columns=[objetivo]
)

y_train_equilibrado = datos_equilibrados[objetivo]


# =====================================================
# 11. COMPROBAR EL RESULTADO FINAL
# =====================================================

print("\nANTES DEL EQUILIBRADO:")
print(y_train.value_counts())

print("\nDESPUÉS DEL EQUILIBRADO:")
print(y_train_equilibrado.value_counts())

print("\nDATOS DE PRUEBA (SIN MODIFICAR):")
print(y_test.value_counts())

print("\n========== RESULTADO FINAL ==========")

print("Filas de entrenamiento:", len(X_train_equilibrado))
print("Filas de prueba:", len(X_test))
print("Columnas finales:", X_train_equilibrado.shape[1])

print("\n¡REDUCCIÓN Y EQUILIBRADO COMPLETADOS!")